In [5]:
import os
import fitz  # PyMuPDF

# =========================================================================
# 1. 【定義階段】空間幾何腳註焊接器 (Spatial Footnote Linker)
# =========================================================================
def spatial_footnote_text_extractor(pdf_path):
    """
    輸入 PDF 絕對路徑，回傳內含正文與頁尾腳註完美焊接的標準 RAG Chunks 列表。
    """
    if not os.path.exists(pdf_path):
        print(f"⚠️ 提示：找不到目標檔案: {pdf_path}")
        return []
        
    doc = fitz.open(pdf_path)
    page_chunks = []
    
    for page_num in range(len(doc)):
        page = doc[page_num]
        page_height = page.rect.height
        footnote_boundary = page_height * 0.85  # 鎖定底部 15% 空間
        
        blocks = page.get_text("blocks")
        main_body_text = ""
        footnote_text = ""
        
        for b in blocks:
            x0, y0, x1, y1, text_content, block_no, block_type = b
            clean_text = text_content.strip()
            if not clean_text:
                continue
                
            if y0 > footnote_boundary:
                footnote_text += f" [Footnote Detail: {clean_text}]"
            else:
                main_body_text += f" {clean_text}"
        
        if footnote_text:
            final_content = f"{main_body_text}\n\n📢 [Linked Footnote For This Context]: {footnote_text}"
        else:
            final_content = main_body_text
            
        page_chunks.append({
            "page_content": f"[SOURCE: {os.path.basename(pdf_path)}_P{page_num+1}]\n{final_content}",
            "metadata": {
                "source_file": os.path.basename(pdf_path),
                "page": page_num + 1,
                "data_type": "text_with_welded_footnotes"
            }
        })
        
    return page_chunks

# =========================================================================
# 2. 【執行階段】使用 Renku 環境絕對路徑進行批量處理
# =========================================================================

# 絕對路徑資料夾位置
pdf_folder_path = "/home/renku/work/Durham-Hackathon-2026-w1t2/Sandy/real_pdf_files"
all_cleaned_chunks = []

print("🔍 正在初始化環境並驗證絕對路徑...")

if not os.path.exists(pdf_folder_path):
    print(f"❌ 錯誤：找不到 Renku 絕對路徑：{pdf_folder_path}，請確認主辦單位的資料夾名稱或路徑！")
else:
    # 自動抓取該資料夾下所有的 PDF 檔案
    pdf_files = [f for f in os.listdir(pdf_folder_path) if f.endswith('.pdf')]
    print(f"專案目錄確認成功！🎯")
    print(f"🕵️‍♂️ 偵測到 {len(pdf_files)} 個 PDF 目標檔案，開始進行空間幾何腳註焊接...")
    
    for pdf_name in pdf_files:
        # 拼接成完整的實體絕對路徑
        full_absolute_path = os.path.join(pdf_folder_path, pdf_name)
        
        # 呼叫上方剛剛定義好的焊接函數（這次絕對不會找不到定義了！）
        chunks = spatial_footnote_text_extractor(full_absolute_path)
        all_cleaned_chunks.extend(chunks)
        
    print(f"\n🎉 完美收工！共計成功清洗並焊接了 {len(all_cleaned_chunks)} 個文本網格（Chunks）！")
    print("🚀 變數 `all_cleaned_chunks` 已就緒，文字組可以直接拿去餵向量資料庫了！")

🔍 正在初始化環境並驗證絕對路徑...
專案目錄確認成功！🎯
🕵️‍♂️ 偵測到 4 個 PDF 目標檔案，開始進行空間幾何腳註焊接...

🎉 完美收工！共計成功清洗並焊接了 527 個文本網格（Chunks）！
🚀 變數 `all_cleaned_chunks` 已就緒，文字組可以直接拿去餵向量資料庫了！
